# Track-map outline spike — derive the Tempelhof circuit from GPS (multi-car)

**Goal:** a clean, complete *GPS-derived SVG* track outline for the Race-Day Companion.
Aggregate several cars' 20 Hz GPS, pick the single cleanest **complete** lap across the field
for the drawn line, and size the viewBox from ALL their points so no car can ever fall
off-screen. Emit an SVG `<path>` + the projection transform to `track_outline.json`.

**Why GPS, not the circuit PDF:** cars and track share one coordinate source, so live car
dots sit on the track by construction — no manual georeferencing.

Run top-to-bottom in Colab Enterprise (needs read access to gs://class-demo).

In [ ]:
%pip install -q gcsfs pyarrow pandas numpy matplotlib

In [ ]:
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth

_, PROJECT_ID = google.auth.default()
print("Project:", PROJECT_ID)

# Cars to aggregate. More cars = a more complete extent and a likelier pristine lap, but slower.
# These are strong full-distance R10 runners; edit freely (or extend to all 22).
CARS = [13, 94, 37, 1, 17, 9, 11, 2, 25, 8]
TELEM_DIR = "gs://class-demo/formula-e/berlin_2024/r10/telemetry/"  # 20 Hz, partitioned by car_number
GREEN_FLAG = pd.Timestamp("2024-05-12 13:04:05.726")  # R10 green flag (UTC); repo-wide constant

## 1. Load + filter each car's GPS (race only)

In [ ]:
def load_latlng(car):
    """Return (lat, lng) arrays for one car, race-only, finite, non-zero."""
    try:
        d = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", car)], engine="pyarrow")
        if d.empty:
            d = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", str(car))], engine="pyarrow")
    except Exception as e:
        print(f"  car {car}: partitioned read failed ({e}); skipping")
        return None
    tcol = next((c for c in d.columns if re.search(r"ts|time|utc|wall|elapsed", c, re.I)), None)
    if tcol:
        d = d.sort_values(tcol)
        d = d[pd.to_datetime(d[tcol]) >= GREEN_FLAG]
    lat = d["tv_gps_lat"].to_numpy(float)
    lng = d["tv_gps_long"].to_numpy(float)
    ok = np.isfinite(lat) & np.isfinite(lng) & (lat != 0) & (lng != 0)
    return lat[ok], lng[ok]

raw = {}
for car in CARS:
    ll = load_latlng(car)
    if ll is not None and len(ll[0]) > 1000:
        raw[car] = ll
        print(f"  car {car}: {len(ll[0])} points")
assert raw, "no car telemetry loaded"
print(f"loaded {len(raw)} cars")

## 2. Project all cars with ONE shared transform (local equirectangular, metres)
Origin is the mean of every loaded point, so all cars and the saved transform agree. The
live UI uses this exact transform for car dots.

In [ ]:
all_lat = np.concatenate([ll[0] for ll in raw.values()])
all_lng = np.concatenate([ll[1] for ll in raw.values()])
lat0, lng0 = float(all_lat.mean()), float(all_lng.mean())
M_PER_DEG_LAT = 110540.0
M_PER_DEG_LNG = 111320.0 * math.cos(math.radians(lat0))

def project(lat, lng):
    return (lng - lng0) * M_PER_DEG_LNG, (lat - lat0) * M_PER_DEG_LAT

xy = {car: project(lat, lng) for car, (lat, lng) in raw.items()}
X_all = np.concatenate([x for x, _ in xy.values()])
Y_all = np.concatenate([y for _, y in xy.values()])

plt.figure(figsize=(8, 8))
for x, y in xy.values():
    plt.plot(x, y, lw=0.2)
plt.gca().set_aspect("equal")
plt.title(f"All GPS points, {len(raw)} cars overlaid = the circuit")
plt.show()

## 3. Pick the single cleanest COMPLETE lap across all cars
Per car, detect laps (trace passing a mid-track anchor), keep full-length laps with no GPS
dropout (small max jump between 20 Hz samples), and choose the globally cleanest one. With a
whole field of laps, there is always a pristine one — no dependence on any single car's luck.

In [ ]:
def laps_of(x, y):
    mid = len(x) // 2
    d = np.hypot(x - x[mid], y - y[mid])
    near = d < 25.0
    cross = np.where((~near[:-1]) & near[1:])[0] + 1
    return [(int(cross[i]), int(cross[i + 1])) for i in range(len(cross) - 1)]

def seglen(x, y, a, b):
    return float(np.hypot(np.diff(x[a:b]), np.diff(y[a:b])).sum()) if b - a > 2 else 0.0
def maxjump(x, y, a, b):
    return float(np.hypot(np.diff(x[a:b]), np.diff(y[a:b])).max()) if b - a > 2 else 1e9

candidates = []  # (maxjump, length, car, a, b)
for car, (x, y) in xy.items():
    for (a, b) in laps_of(x, y):
        L, J = seglen(x, y, a, b), maxjump(x, y, a, b)
        if L > 1800 and J < 25.0:
            candidates.append((J, L, car, a, b))

if candidates:
    J, L, car, a, b = min(candidates, key=lambda t: t[0])   # cleanest complete lap in the field
    print(f"chose car {car} lap [{a}:{b}] — {L:.0f} m, max jump {J:.1f} m "
          f"({len(candidates)} clean laps available)")
else:  # fallback: longest detected lap anywhere
    best = max(((seglen(*xy[c], a, b), c, a, b) for c in xy for (a, b) in laps_of(*xy[c])),
               default=(0, CARS[0], 0, len(X_all) - 1))
    L, car, a, b = best
    print(f"no pristine lap found; fell back to car {car} lap [{a}:{b}] — {L:.0f} m")
lx, ly = xy[car][0][a:b + 1], xy[car][1][a:b + 1]

plt.figure(figsize=(8, 8))
plt.plot(lx, ly, lw=1.0)
plt.gca().set_aspect("equal")
plt.title(f"Chosen lap — car {car}")
plt.show()

## 4. Build the SVG path + viewBox (extent from ALL cars)
Path = the chosen clean lap. Extent (viewBox + transform) = the aggregate of EVERY car's
points, so any live car projects inside the viewBox. y is flipped (SVG y grows downward).

In [ ]:
N = 400
idx = np.linspace(0, len(lx) - 1, min(N, len(lx))).astype(int)
sx, sy = lx[idx], ly[idx]

pad = 30.0
minx, maxx = float(X_all.min()), float(X_all.max())   # aggregate extent (all cars)
miny, maxy = float(Y_all.min()), float(Y_all.max())
W, H = maxx - minx, maxy - miny

def to_svg(px, py):
    return (px - minx) + pad, (maxy - py) + pad   # flip y

pts = [to_svg(px, py) for px, py in zip(sx, sy)]
path = "M " + " L ".join(f"{px:.1f} {py:.1f}" for px, py in pts) + " Z"
viewBox = f"0 0 {W + 2 * pad:.1f} {H + 2 * pad:.1f}"
print("viewBox:", viewBox, "| path points:", len(pts))
print("path head:", path[:120], "...")

plt.figure(figsize=(8, 8))
plt.plot([p[0] for p in pts], [p[1] for p in pts])
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("SVG-space preview (how it will render in the UI)")
plt.show()

In [ ]:
out = {
    "race_id": "berlin_2024_r10",
    "viewBox": viewBox,
    "path": path,
    # Live UI: x=(lng-lng0)*m_per_deg_lng ; y=(lat-lat0)*m_per_deg_lat ;
    #          svg_x=(x-minx)+pad ; svg_y=(maxy-y)+pad
    "transform": {
        "lat0": lat0, "lng0": lng0,
        "m_per_deg_lat": M_PER_DEG_LAT, "m_per_deg_lng": M_PER_DEG_LNG,
        "minx": minx, "maxy": maxy, "pad": pad,
    },
}
with open("track_outline.json", "w") as f:
    json.dump(out, f, indent=2)
print("wrote track_outline.json")

## Done
Drop `track_outline.json` into `frontend/static/`. The outline is the cleanest complete lap
in the field; the viewBox spans every car's points, so nothing flies off-screen. Add more
cars to `CARS` for an even fuller extent.